# Thesis results: AMM sandwich profitability

This notebook is the thesis-facing analysis layer for simulator CSV outputs. It is intentionally tolerant of older result files: if a CSV was produced before the current schema, the setup cell fills derived columns and marks the dataset as `legacy_schema`.

Recommended fresh inputs:

- `results/sweep.csv` from `cargo run --bin mev-sim -- -c configs/sweep_liquidity.toml -o results/sweep.csv --parallel`
- `results/zhou_vs_numerical.csv` from `cargo run -p simulator --example bench_zhou_vs_numerical -- -o results/zhou_vs_numerical.csv`
- `results/real_pool_comparison.csv` from `cargo run -p simulator --bin compare_real_pool -- -c configs/sweep_real_pool.toml -o results/real_pool_comparison.csv --parallel`
- optional `results/historical_cpmm_candidates.csv` from decoded historical Raydium CPMM swaps


## 1. Setup and schema normalization


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

RECOMMENDED_MAIN_COLUMNS = {
    "strategy",
    "attack_status",
    "attack_feasible",
    "victim_reverted",
    "victim_size_bps_of_reserve",
    "frontrun_size_bps_of_reserve",
    "net_profit_bps_of_frontrun",
    "victim_loss_bps_of_fair_out",
    "tx_cost_per_leg",
}


def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "Cargo.toml").exists() and (candidate / "simulator").exists():
            return candidate
    raise RuntimeError("Cannot find repo root. Run the notebook from this repository.")


ROOT = find_repo_root()
RESULTS = ROOT / "results"


def first_existing(*names):
    for name in names:
        path = RESULTS / name
        if path.exists():
            return path
    return None


def read_csv(path):
    if path is None:
        return pd.DataFrame()
    return pd.read_csv(path)


def as_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def bps(num, den):
    den = pd.to_numeric(den, errors="coerce").replace(0, np.nan)
    return (pd.to_numeric(num, errors="coerce") * 10_000 / den).replace([np.inf, -np.inf], np.nan)


def normalize_main(df, dataset_name):
    df = df.copy()
    if df.empty:
        df.attrs["dataset_name"] = dataset_name
        df.attrs["legacy_schema"] = False
        df.attrs["missing_recommended_columns"] = sorted(RECOMMENDED_MAIN_COLUMNS)
        return df

    original_columns = set(df.columns)
    df.attrs["dataset_name"] = dataset_name
    df.attrs["legacy_schema"] = not RECOMMENDED_MAIN_COLUMNS.issubset(original_columns)
    df.attrs["missing_recommended_columns"] = sorted(RECOMMENDED_MAIN_COLUMNS - original_columns)

    if "strategy" not in df:
        df["strategy"] = "legacy_closed_form"
    if "pool_label" not in df:
        df["pool_label"] = np.where(df.get("source", "").astype(str).str.contains("real|raydium", case=False, na=False), "unknown_real_pool", "synthetic")
    if "attack_profitable" in df:
        df["attack_profitable"] = as_bool(df["attack_profitable"])
    else:
        df["attack_profitable"] = pd.to_numeric(df.get("attacker_net_profit", 0), errors="coerce") > 0
    if "attack_feasible" not in df:
        if {"victim_extra_slippage_bps", "victim_slippage_tolerance_bps"}.issubset(df.columns):
            df["attack_feasible"] = pd.to_numeric(df["victim_extra_slippage_bps"], errors="coerce") <= pd.to_numeric(df["victim_slippage_tolerance_bps"], errors="coerce")
        else:
            df["attack_feasible"] = True
    else:
        df["attack_feasible"] = as_bool(df["attack_feasible"])
    if "victim_reverted" not in df:
        df["victim_reverted"] = ~df["attack_feasible"]
    else:
        df["victim_reverted"] = as_bool(df["victim_reverted"])
    if "attack_status" not in df:
        df["attack_status"] = np.select(
            [df["frontrun_amount"].fillna(0).eq(0), df["attack_profitable"]],
            ["no_profitable_attack", "executed"],
            default="executed",
        )
    if "victim_size_bps_of_reserve" not in df and {"victim_amount", "pool_reserve_a"}.issubset(df.columns):
        df["victim_size_bps_of_reserve"] = bps(df["victim_amount"], df["pool_reserve_a"]).round()
    if "frontrun_size_bps_of_reserve" not in df and {"frontrun_amount", "pool_reserve_a"}.issubset(df.columns):
        df["frontrun_size_bps_of_reserve"] = bps(df["frontrun_amount"], df["pool_reserve_a"]).round()
    if "victim_loss_bps_of_fair_out" not in df and {"victim_loss_absolute", "victim_amount_out_no_attack"}.issubset(df.columns):
        df["victim_loss_bps_of_fair_out"] = bps(df["victim_loss_absolute"], df["victim_amount_out_no_attack"]).round()
    if "tx_cost_per_leg" not in df and "tx_cost_total" in df:
        df["tx_cost_per_leg"] = pd.to_numeric(df["tx_cost_total"], errors="coerce") / 2
    if "net_profit_bps_of_frontrun" not in df and {"attacker_net_profit", "frontrun_amount"}.issubset(df.columns):
        df["net_profit_bps_of_frontrun"] = bps(df["attacker_net_profit"], df["frontrun_amount"])

    for col in [
        "pool_fee_bps",
        "victim_size_bps_of_reserve",
        "frontrun_size_bps_of_reserve",
        "victim_loss_bps_of_fair_out",
        "victim_extra_slippage_bps",
        "victim_slippage_tolerance_bps",
        "attacker_net_profit",
        "attacker_gross_profit",
        "tx_cost_per_leg",
        "tx_cost_total",
    ]:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["attack_realized"] = df["attack_profitable"] & df["attack_feasible"]
    return df


main_path = first_existing("sweep.csv", "output.csv")
zhou_path = first_existing("zhou_vs_numerical.csv")
real_cmp_path = first_existing("real_pool_comparison.csv", "sweep_real_pool.csv")
historical_path = first_existing("historical_cpmm_candidates.csv")

main = normalize_main(read_csv(main_path), "main_sweep")
real_cmp = normalize_main(read_csv(real_cmp_path), "real_pool")
zhou = read_csv(zhou_path)
historical = read_csv(historical_path)

status = pd.DataFrame([
    {
        "dataset": "main",
        "path": str(main_path.relative_to(ROOT)) if main_path else None,
        "rows": len(main),
        "columns": len(main.columns),
        "legacy_schema": main.attrs.get("legacy_schema", False),
        "missing_recommended_columns": ", ".join(main.attrs.get("missing_recommended_columns", [])),
    },
    {
        "dataset": "zhou_vs_numerical",
        "path": str(zhou_path.relative_to(ROOT)) if zhou_path else None,
        "rows": len(zhou),
        "columns": len(zhou.columns),
        "legacy_schema": False,
        "missing_recommended_columns": "",
    },
    {
        "dataset": "real_pool",
        "path": str(real_cmp_path.relative_to(ROOT)) if real_cmp_path else None,
        "rows": len(real_cmp),
        "columns": len(real_cmp.columns),
        "legacy_schema": real_cmp.attrs.get("legacy_schema", False),
        "missing_recommended_columns": ", ".join(real_cmp.attrs.get("missing_recommended_columns", [])),
    },
    {
        "dataset": "historical_cpmm",
        "path": str(historical_path.relative_to(ROOT)) if historical_path else None,
        "rows": len(historical),
        "columns": len(historical.columns),
        "legacy_schema": False,
        "missing_recommended_columns": "",
    },
])

display(status)
if status["legacy_schema"].any():
    display(Markdown("> Some CSVs are old-schema files. The notebook filled derived columns so the analysis can run, but regenerate fresh CSVs before using final thesis numbers."))


## 2. Main simulator results


In [ ]:
if main.empty:
    display(Markdown("No main simulator CSV found. Generate `results/sweep.csv` first."))
else:
    summary = (
        main.groupby(["source", "strategy", "attack_status"], dropna=False)
        .agg(
            rows=("attack_status", "size"),
            profitable_rate=("attack_profitable", "mean"),
            feasible_rate=("attack_feasible", "mean"),
            realized_rate=("attack_realized", "mean"),
            median_net_profit=("attacker_net_profit", "median"),
            p90_net_profit=("attacker_net_profit", lambda s: s.quantile(0.9)),
            median_victim_loss_bps=("victim_loss_bps_of_fair_out", "median"),
        )
        .reset_index()
        .sort_values(["realized_rate", "median_net_profit"], ascending=False)
    )
    display(summary)

    realized = main[main["attack_realized"]]
    best = realized.sort_values("attacker_net_profit", ascending=False).head(10)
    display(Markdown(f"**Core result:** {len(realized)} / {len(main)} rows are both profitable and feasible under the victim slippage constraint."))
    display(best[[
        "source", "pool_label", "strategy", "pool_fee_bps", "victim_amount", "victim_size_bps_of_reserve",
        "victim_slippage_tolerance_bps", "frontrun_amount", "attacker_net_profit", "victim_loss_bps_of_fair_out",
        "tx_cost_per_leg",
    ]])


## 3. Profitability surface


In [ ]:
if main.empty or not {"pool_fee_bps", "victim_size_bps_of_reserve", "attack_realized"}.issubset(main.columns):
    display(Markdown("Missing columns for profitability surface."))
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    success = main.pivot_table(
        index="victim_size_bps_of_reserve",
        columns="pool_fee_bps",
        values="attack_realized",
        aggfunc="mean",
    ).sort_index()
    sns.heatmap(success, annot=True, fmt=".2f", cmap="viridis", cbar_kws={"label": "profitable + feasible rate"}, ax=axes[0])
    axes[0].set_title("Attack success rate")
    axes[0].set_xlabel("pool fee [bps]")
    axes[0].set_ylabel("victim size [bps of reserve_in]")

    profit = main.pivot_table(
        index="victim_size_bps_of_reserve",
        columns="pool_fee_bps",
        values="attacker_net_profit",
        aggfunc="median",
    ).sort_index()
    sns.heatmap(profit, annot=True, fmt=".0f", cmap="mako", cbar_kws={"label": "median net profit"}, ax=axes[1])
    axes[1].set_title("Median attacker net profit")
    axes[1].set_xlabel("pool fee [bps]")
    axes[1].set_ylabel("")

    plt.tight_layout()


## 4. Cost and slippage sensitivity


In [ ]:
if main.empty:
    display(Markdown("No main simulator data."))
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    if "tx_cost_per_leg" in main:
        cost_summary = (
            main.groupby("tx_cost_per_leg")
            .agg(
                rows=("tx_cost_per_leg", "size"),
                realized_rate=("attack_realized", "mean"),
                median_net_profit=("attacker_net_profit", "median"),
            )
            .reset_index()
            .sort_values("tx_cost_per_leg")
        )
        display(cost_summary)
        sns.lineplot(data=cost_summary, x="tx_cost_per_leg", y="realized_rate", marker="o", ax=axes[0])
        axes[0].set_title("Realized attack rate vs per-leg transaction cost")
        axes[0].set_xlabel("attacker cost per leg [token_in units]")
        axes[0].set_ylabel("profitable + feasible rate")

    if "victim_slippage_tolerance_bps" in main:
        slip_summary = (
            main.groupby("victim_slippage_tolerance_bps")
            .agg(
                rows=("victim_slippage_tolerance_bps", "size"),
                feasible_rate=("attack_feasible", "mean"),
                realized_rate=("attack_realized", "mean"),
                median_victim_loss_bps=("victim_loss_bps_of_fair_out", "median"),
            )
            .reset_index()
            .sort_values("victim_slippage_tolerance_bps")
        )
        display(slip_summary)
        sns.lineplot(data=slip_summary, x="victim_slippage_tolerance_bps", y="realized_rate", marker="o", ax=axes[1])
        axes[1].set_title("Realized attack rate vs victim slippage tolerance")
        axes[1].set_xlabel("victim slippage tolerance [bps]")
        axes[1].set_ylabel("profitable + feasible rate")

    plt.tight_layout()


## 5. Closed-form Zhou baseline vs numerical optimizer


In [ ]:
if zhou.empty:
    display(Markdown("No `zhou_vs_numerical.csv` found. Generate it before using optimizer-comparison claims."))
else:
    for col in ["fee_bps", "victim_bps_of_reserve", "tx_cost_per_leg", "profit_gap_abs", "frontrun_gap_abs", "numerical_grid_profit_gap"]:
        if col in zhou:
            zhou[col] = pd.to_numeric(zhou[col], errors="coerce")

    display(
        zhou.groupby(["liquidity_depth_label", "fee_bps"])
        .agg(
            rows=("fee_bps", "size"),
            median_profit_gap=("profit_gap_abs", "median"),
            p90_profit_gap=("profit_gap_abs", lambda s: s.quantile(0.9)),
            median_frontrun_gap=("frontrun_gap_abs", "median"),
            median_numerical_grid_gap=("numerical_grid_profit_gap", "median"),
        )
        .reset_index()
        .sort_values(["liquidity_depth_label", "fee_bps"])
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.lineplot(data=zhou, x="fee_bps", y="profit_gap_abs", hue="liquidity_depth_label", estimator="median", errorbar=None, marker="o", ax=axes[0])
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("Numerical profit minus closed-form profit")
    axes[0].set_xlabel("fee [bps]")
    axes[0].set_ylabel("median profit gap")

    sns.lineplot(data=zhou, x="fee_bps", y="frontrun_gap_abs", hue="liquidity_depth_label", estimator="median", errorbar=None, marker="o", ax=axes[1])
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Numerical frontrun minus closed-form frontrun")
    axes[1].set_xlabel("fee [bps]")
    axes[1].set_ylabel("median frontrun gap")
    plt.tight_layout()


## 6. Raydium CPMM snapshot comparison


In [ ]:
if real_cmp.empty:
    display(Markdown("No real-pool CSV found. Generate `results/real_pool_comparison.csv` for the Raydium CPMM snapshot section."))
else:
    display(
        real_cmp.groupby(["source", "pool_label", "strategy"], dropna=False)
        .agg(
            rows=("source", "size"),
            profitable_rate=("attack_profitable", "mean"),
            feasible_rate=("attack_feasible", "mean"),
            realized_rate=("attack_realized", "mean"),
            median_net_profit=("attacker_net_profit", "median"),
            median_victim_loss_bps=("victim_loss_bps_of_fair_out", "median"),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=real_cmp, x="source", y="attack_realized", hue="strategy", errorbar=None, ax=axes[0])
    axes[0].set_title("Realized attack rate on snapshot-derived scenarios")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("profitable + feasible rate")
    axes[0].tick_params(axis="x", rotation=15)

    sns.scatterplot(
        data=real_cmp,
        x="victim_size_bps_of_reserve",
        y="attacker_net_profit",
        hue="source",
        style="strategy",
        ax=axes[1],
    )
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Net profit by victim size")
    axes[1].set_xlabel("victim size [bps of reserve_in]")
    axes[1].set_ylabel("attacker net profit")
    plt.tight_layout()


## 7. Historical Raydium CPMM candidates


In [ ]:
if historical.empty:
    display(Markdown(
        "No `historical_cpmm_candidates.csv` found yet. This is expected until transaction collection/decoding/pre-state reconstruction exists. "
        "Once generated, this section should become the empirical counterfactual part of the thesis."
    ))
else:
    for col in ["attack_profitable", "attack_feasible", "attack_realized", "below_candidate_threshold"]:
        if col in historical:
            historical[col] = as_bool(historical[col])
    if "attack_realized" not in historical and {"attack_profitable", "attack_feasible"}.issubset(historical.columns):
        historical["attack_realized"] = historical["attack_profitable"] & historical["attack_feasible"]

    display(
        historical.groupby(["pool_label", "inclusion_reason"], dropna=False)
        .agg(
            rows=("pool_label", "size"),
            profitable_rate=("attack_profitable", "mean"),
            feasible_rate=("attack_feasible", "mean"),
            realized_rate=("attack_realized", "mean"),
            median_net_profit=("net_profit", "median"),
        )
        .reset_index()
    )

    plt.figure(figsize=(9, 5))
    sns.histplot(data=historical, x="net_profit", hue="attack_realized", bins=40, element="step")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Historical candidate counterfactual net profit")
    plt.xlabel("net profit [token_in units]")
    plt.tight_layout()


## 8. Thesis takeaways draft


In [ ]:
lines = []
if not main.empty:
    realized_rate = main["attack_realized"].mean()
    profitable_rate = main["attack_profitable"].mean()
    feasible_rate = main["attack_feasible"].mean()
    lines.append(f"- In the synthetic sweep, {profitable_rate:.1%} rows are profitable before slippage filtering, {feasible_rate:.1%} are feasible under victim slippage, and {realized_rate:.1%} are both.")
    if "victim_size_bps_of_reserve" in main:
        by_size = main.groupby("victim_size_bps_of_reserve")["attack_realized"].mean().sort_index()
        if not by_size.empty:
            lines.append(f"- Realized attack rate rises from {by_size.iloc[0]:.1%} at {by_size.index[0]:.0f} bps victim size to {by_size.iloc[-1]:.1%} at {by_size.index[-1]:.0f} bps.")
    if "tx_cost_per_leg" in main:
        by_cost = main.groupby("tx_cost_per_leg")["attack_realized"].mean().sort_index()
        if len(by_cost) > 1:
            lines.append(f"- Transaction costs materially shift feasibility: realized rate changes from {by_cost.iloc[0]:.1%} at lowest modeled cost to {by_cost.iloc[-1]:.1%} at highest modeled cost.")
if not zhou.empty:
    median_gap = pd.to_numeric(zhou["profit_gap_abs"], errors="coerce").median()
    lines.append(f"- The numerical optimizer median profit gap versus the closed-form baseline is {median_gap:.0f} token-in units in the benchmark grid, so the baseline should be treated as a reference, not the final estimator.")
if not real_cmp.empty:
    real_rate = real_cmp["attack_realized"].mean()
    lines.append(f"- Snapshot-derived Raydium CPMM scenarios have {real_rate:.1%} realized attack rate under the current parameter grid.")
if historical.empty:
    lines.append("- Historical Raydium CPMM/CLMM evidence is still the main missing empirical piece: transaction selection, decoding, and pre-state reconstruction must be completed before final conclusions.")

display(Markdown("\n".join(lines) if lines else "No data loaded yet."))
